In [1]:
# =============================================================
# DAY 4 — From Model to Farmer Tool
# Author  : Daniel Oyanogbezina
# Goal    : Load Day 3 model and build interactive farmer tool
# =============================================================

import pickle
import pandas as pd
import numpy as np
import warnings
import os
warnings.filterwarnings('ignore')

print("=" * 55)
print("  Day 4 — From Model to Farmer Tool")
print("  Author: Daniel Oyanogbezina")
print("=" * 55)

# --- Load the model pipeline saved on Day 3 ---
print("\nLoading saved model from Day 3...")

try:
    with open('models/yield_predictor_v1.pkl', 'rb') as f:
        model = pickle.load(f)

    with open('models/feature_cols.pkl', 'rb') as f:
        feature_cols = pickle.load(f)

    print("Model loaded successfully!")
    print(f"Features the model knows: {len(feature_cols)}")
    print("\nFeatures list:")
    for i, col in enumerate(feature_cols, 1):
        print(f"  {i:2}. {col}")

except FileNotFoundError as e:
    print(f"Error: {e}")
    print("Run Day 3 Cell 8 first to save the model!")

  Day 4 — From Model to Farmer Tool
  Author: Daniel Oyanogbezina

Loading saved model from Day 3...
Model loaded successfully!
Features the model knows: 12

Features list:
   1. rainfall_mm
   2. soil_ph
   3. fertilizer_kg
   4. improved_seeds
   5. irrigation
   6. farm_size_ha
   7. crop_Cassava
   8. crop_Yam
   9. crop_Rice
  10. crop_Maize
  11. crop_Groundnut
  12. crop_Cocoa


In [2]:
# What did our Day 3 model actually learn?
# Let's inspect it before building on top of it

print("INSPECTING THE SAVED MODEL")
print("=" * 55)

# The pipeline has two steps
steps = model.named_steps
print(f"\nPipeline steps: {list(steps.keys())}")

# Check the imputer
imputer = steps['imputer']
print(f"\nImputer strategy : {imputer.strategy}")
print("(fills NaN with the median value of each column)")

# Check the Random Forest
rf = steps['model']
print(f"\nRandom Forest details:")
print(f"  Number of trees : {rf.n_estimators}")
print(f"  Max depth       : {rf.max_depth}")
print(f"  Features used   : {rf.n_features_in_}")

# Quick test prediction
test_input = pd.DataFrame([{col: 0 for col in feature_cols}])
test_input['rainfall_mm']   = 1200
test_input['soil_ph']       = 6.2
test_input['fertilizer_kg'] = 100
test_input['improved_seeds']= 1
test_input['irrigation']    = 1
test_input['farm_size_ha']  = 4.0
test_input['crop_Cassava']  = 1

test_pred = model.predict(test_input)[0]

print(f"\nTest prediction:")
print(f"  Cassava farm, 1200mm rain, full inputs, 4ha")
print(f"  Predicted yield: {test_pred:.1f} tons")
print(f"\nModel is working correctly and ready for Day 4!")

INSPECTING THE SAVED MODEL

Pipeline steps: ['imputer', 'model']

Imputer strategy : median
(fills NaN with the median value of each column)

Random Forest details:
  Number of trees : 100
  Max depth       : 8
  Features used   : 12

Test prediction:
  Cassava farm, 1200mm rain, full inputs, 4ha
  Predicted yield: 25.7 tons

Model is working correctly and ready for Day 4!


In [3]:
# =============================================================
# ADVISORY KNOWLEDGE BASE
# This is where YOUR agroforestry expertise becomes code
# Every piece of advice below comes from real agronomy
# =============================================================

CROP_ADVICE = {
    'Cassava': {
        'season':        'Plant: March-April or August-September',
        'soil_ph':       'Best pH: 5.5 - 6.5',
        'spacing':       'Spacing: 1m x 1m (10,000 stands/ha)',
        'harvest':       'Harvest: 9-12 months after planting',
        'fertilizer':    'Apply NPK 15-15-15 at 6 weeks after planting',
        'water_need':    'Moderate — tolerates short dry spells',
        'disease_watch': 'Watch for: Cassava Mosaic, Brown Streak',
        'market_tip':    'Best price: Dec-Feb when supply is low',
        'income_range':  (40000, 80000),   # Naira per ton
    },
    'Yam': {
        'season':        'Plant: January-March (before first rains)',
        'soil_ph':       'Best pH: 5.5 - 6.0',
        'spacing':       'Spacing: 1m x 1m on ridges or mounds',
        'harvest':       'Harvest: 7-10 months after planting',
        'fertilizer':    'Apply NPK at planting + top-dress at 8 weeks',
        'water_need':    'High in first 3 months — ensure good rainfall',
        'disease_watch': 'Watch for: Yam Anthracnose, Dry Rot',
        'market_tip':    'Best price: August-October (New Yam Festival)',
        'income_range':  (200000, 400000),
    },
    'Maize': {
        'season':        'Plant: March-April (1st) or July-August (2nd)',
        'soil_ph':       'Best pH: 5.8 - 7.0',
        'spacing':       'Spacing: 75cm x 25cm (2 seeds per hole)',
        'harvest':       'Harvest: 90-120 days after planting',
        'fertilizer':    'NPK at planting + Urea top-dress at 6 weeks',
        'water_need':    'Critical during tasselling — do not allow drought',
        'disease_watch': 'Watch for: Fall Armyworm (inspect weekly)',
        'market_tip':    'Best price: November-January (off-season)',
        'income_range':  (150000, 250000),
    },
    'Rice': {
        'season':        'Plant: May-June (wet season)',
        'soil_ph':       'Best pH: 5.5 - 6.5',
        'spacing':       'Spacing: 20cm x 20cm (transplanting)',
        'harvest':       'Harvest: 100-140 days after planting',
        'fertilizer':    'Apply Urea in splits (planting + tillering)',
        'water_need':    'Very high — paddy needs standing water',
        'disease_watch': 'Watch for: Blast, Bacterial Leaf Blight',
        'market_tip':    'Best price: January-March (dry season)',
        'income_range':  (180000, 320000),
    },
    'Groundnut': {
        'season':        'Plant: April-May with early rains',
        'soil_ph':       'Best pH: 6.0 - 6.5',
        'spacing':       'Spacing: 45cm x 15cm',
        'harvest':       'Harvest: 90-120 days after planting',
        'fertilizer':    'Low nitrogen needed (fixes own nitrogen)',
        'water_need':    'Moderate — sensitive to waterlogging',
        'disease_watch': 'Watch for: Rosette virus, Leaf Spot',
        'market_tip':    'Best price: November-December',
        'income_range':  (120000, 220000),
    },
    'Cocoa': {
        'season':        'Plant: April-June (main season)',
        'soil_ph':       'Best pH: 6.0 - 7.0',
        'spacing':       'Spacing: 3m x 3m under shade trees',
        'harvest':       'First harvest: 3-5 years after planting',
        'fertilizer':    'Apply annually — NPK + micro-nutrients',
        'water_need':    'High and consistent — not drought tolerant',
        'disease_watch': 'Watch for: Black Pod, Swollen Shoot Virus',
        'market_tip':    'Price linked to global market — sell through co-ops',
        'income_range':  (800000, 1500000),
    },
}

# Rainfall categories for advice
def rainfall_category(mm):
    if mm < 600:   return 'very_dry'
    if mm < 900:   return 'dry'
    if mm < 1200:  return 'moderate'
    if mm < 1600:  return 'good'
    return 'very_wet'

RAINFALL_ADVICE = {
    'very_dry':  'Irrigation is ESSENTIAL. Consider drought-resistant varieties.',
    'dry':       'Supplement with irrigation if possible. Mulch to retain moisture.',
    'moderate':  'Good for most crops. Monitor soil moisture weekly.',
    'good':      'Excellent rainfall. Focus on disease prevention.',
    'very_wet':  'High flood and fungal disease risk. Ensure good drainage.',
}

# Soil pH advice
def soil_ph_advice(ph):
    if ph < 5.0:   return 'Very acidic — apply lime to raise pH before planting'
    if ph < 5.5:   return 'Slightly acidic — lime application recommended'
    if ph <= 6.5:  return 'Optimal range — most crops will thrive'
    if ph <= 7.0:  return 'Slightly alkaline — suitable for maize and groundnut'
    return 'Too alkaline — sulphur application may be needed'

print("Advisory knowledge base loaded!")
print(f"Crops covered: {list(CROP_ADVICE.keys())}")
print("\nYour agroforestry expertise is now encoded as AI advice.")

Advisory knowledge base loaded!
Crops covered: ['Cassava', 'Yam', 'Maize', 'Rice', 'Groundnut', 'Cocoa']

Your agroforestry expertise is now encoded as AI advice.


In [4]:
# =============================================================
# CORE PREDICTION ENGINE
# Takes farmer inputs -> returns prediction + full advice
# =============================================================

def predict_and_advise(crop, rainfall_mm, soil_ph,
                       fertilizer_kg, improved_seeds,
                       irrigation, farm_size_ha):
    """
    Main function for the farmer advisory tool.

    Returns a complete advisory report including:
    - Predicted yield
    - Estimated income
    - Personalised farming advice
    - Risk flags
    - Market timing tips
    """

    # --- Input validation ---
    crop = crop.strip().title()
    if crop not in CROP_ADVICE:
        return {
            'error': f"Crop '{crop}' not in database. "
                     f"Available: {list(CROP_ADVICE.keys())}"
        }

    if not (0 < farm_size_ha <= 50):
        return {'error': 'Farm size must be between 0.1 and 50 hectares'}

    if not (300 <= rainfall_mm <= 3000):
        return {'error': 'Rainfall must be between 300 and 3000 mm'}

    # --- Build model input ---
    row = {col: 0 for col in feature_cols}
    row['rainfall_mm']    = rainfall_mm
    row['soil_ph']        = soil_ph
    row['fertilizer_kg']  = fertilizer_kg
    row['improved_seeds'] = int(improved_seeds)
    row['irrigation']     = int(irrigation)
    row['farm_size_ha']   = farm_size_ha

    crop_col = f'crop_{crop}'
    if crop_col in row:
        row[crop_col] = 1

    input_df  = pd.DataFrame([row])
    yield_hat = round(max(1.0, model.predict(input_df)[0]), 1)

    # --- Income estimate ---
    advice_db      = CROP_ADVICE[crop]
    price_low, price_high = advice_db['income_range']
    avg_price      = (price_low + price_high) / 2
    income_low     = yield_hat * price_low
    income_high    = yield_hat * price_high
    income_mid     = yield_hat * avg_price

    # --- Risk flags ---
    risks = []
    r_cat = rainfall_category(rainfall_mm)
    if r_cat in ('very_dry', 'dry') and not irrigation:
        risks.append('LOW RAINFALL + NO IRRIGATION: yield at serious risk')
    if soil_ph < 5.5:
        risks.append('LOW SOIL pH: apply agricultural lime before planting')
    if fertilizer_kg < 20:
        risks.append('LOW FERTILIZER: consider minimum of 50 kg/ha NPK')
    if not improved_seeds:
        risks.append('TRADITIONAL SEEDS: switch to improved varieties for +20% yield')
    if r_cat == 'very_wet' and crop == 'Groundnut':
        risks.append('HIGH RAINFALL + GROUNDNUT: ensure ridge planting for drainage')

    # --- What-if comparisons ---
    # Show farmer what changes would improve their yield
    row_improved = row.copy()
    row_improved['improved_seeds'] = 1
    row_improved['fertilizer_kg']  = max(fertilizer_kg, 100)
    row_improved['irrigation']     = 1
    df_improved  = pd.DataFrame([row_improved])
    yield_improved = round(max(1.0, model.predict(df_improved)[0]), 1)
    uplift_pct   = round((yield_improved - yield_hat) / yield_hat * 100, 1)

    return {
        'crop':            crop,
        'farm_size_ha':    farm_size_ha,
        'predicted_yield': yield_hat,
        'income_low':      income_low,
        'income_mid':      income_mid,
        'income_high':     income_high,
        'rainfall_advice': RAINFALL_ADVICE[r_cat],
        'soil_advice':     soil_ph_advice(soil_ph),
        'season':          advice_db['season'],
        'spacing':         advice_db['spacing'],
        'harvest':         advice_db['harvest'],
        'fertilizer_tip':  advice_db['fertilizer'],
        'disease_watch':   advice_db['disease_watch'],
        'market_tip':      advice_db['market_tip'],
        'risks':           risks,
        'yield_if_improved': yield_improved,
        'uplift_pct':      uplift_pct,
    }


def print_advisory_report(result):
    """Prints a clean, readable advisory report for the farmer"""

    if 'error' in result:
        print(f"\nERROR: {result['error']}")
        return

    sep = "=" * 60

    print(f"\n{sep}")
    print(f"  AGRITECH AI — FARMER ADVISORY REPORT")
    print(f"  Powered by Daniel Oyanogbezina's AgriTech AI Platform")
    print(sep)

    print(f"\n  CROP       : {result['crop']}")
    print(f"  FARM SIZE  : {result['farm_size_ha']} hectares")

    print(f"\n{'-'*60}")
    print(f"  YIELD PREDICTION")
    print(f"{'-'*60}")
    print(f"  Predicted yield   : {result['predicted_yield']} tons")
    print(f"  Est. income (low) : N{result['income_low']:>12,.0f}")
    print(f"  Est. income (mid) : N{result['income_mid']:>12,.0f}")
    print(f"  Est. income (high): N{result['income_high']:>12,.0f}")

    print(f"\n{'-'*60}")
    print(f"  PERSONALISED FARMING ADVICE")
    print(f"{'-'*60}")
    print(f"  Season    : {result['season']}")
    print(f"  Spacing   : {result['spacing']}")
    print(f"  Harvest   : {result['harvest']}")
    print(f"  Fertilizer: {result['fertilizer_tip']}")
    print(f"  Water     : {result['rainfall_advice']}")
    print(f"  Soil      : {result['soil_advice']}")
    print(f"  Disease   : {result['disease_watch']}")
    print(f"  Market    : {result['market_tip']}")

    if result['risks']:
        print(f"\n{'-'*60}")
        print(f"  RISK FLAGS  (action needed)")
        print(f"{'-'*60}")
        for r in result['risks']:
            print(f"  ! {r}")

    print(f"\n{'-'*60}")
    print(f"  WHAT IF YOU UPGRADED YOUR INPUTS?")
    print(f"{'-'*60}")
    print(f"  Current predicted yield  : {result['predicted_yield']} tons")
    print(f"  With improved seeds +    ")
    print(f"  more fertilizer +        ")
    print(f"  irrigation               : {result['yield_if_improved']} tons")
    print(f"  Potential uplift         : +{result['uplift_pct']}%")

    print(f"\n{sep}")
    print(f"  Report generated by AgriTech AI Platform")
    print(f"  Author: Daniel Oyanogbezina")
    print(sep)


print("Prediction engine and report printer ready!")

Prediction engine and report printer ready!


In [5]:
# Test the full advisory system with four Nigerian farmers

test_farmers = [
    {
        'label':           'Alhaji Musa — Kano (Maize, dry north)',
        'crop':            'Maize',
        'rainfall_mm':     680,
        'soil_ph':         6.8,
        'fertilizer_kg':   25,
        'improved_seeds':  False,
        'irrigation':      False,
        'farm_size_ha':    2.0,
    },
    {
        'label':           'Mrs Adunola — Oyo (Cassava, south-west)',
        'crop':            'Cassava',
        'rainfall_mm':     1250,
        'soil_ph':         6.1,
        'fertilizer_kg':   85,
        'improved_seeds':  True,
        'irrigation':      False,
        'farm_size_ha':    3.5,
    },
    {
        'label':           'Chukwuemeka — Benue (Yam, middle belt)',
        'crop':            'Yam',
        'rainfall_mm':     1100,
        'soil_ph':         6.0,
        'fertilizer_kg':   150,
        'improved_seeds':  True,
        'irrigation':      True,
        'farm_size_ha':    8.0,
    },
    {
        'label':           'Fatima — Kebbi (Rice, irrigated)',
        'crop':            'Rice',
        'rainfall_mm':     750,
        'soil_ph':         6.5,
        'fertilizer_kg':   100,
        'improved_seeds':  True,
        'irrigation':      True,
        'farm_size_ha':    5.0,
    },
]

for farmer in test_farmers:
    print(f"\n{'*'*60}")
    print(f"  Testing: {farmer['label']}")
    print(f"{'*'*60}")

    result = predict_and_advise(
        crop            = farmer['crop'],
        rainfall_mm     = farmer['rainfall_mm'],
        soil_ph         = farmer['soil_ph'],
        fertilizer_kg   = farmer['fertilizer_kg'],
        improved_seeds  = farmer['improved_seeds'],
        irrigation      = farmer['irrigation'],
        farm_size_ha    = farmer['farm_size_ha'],
    )

    print_advisory_report(result)
    


************************************************************
  Testing: Alhaji Musa — Kano (Maize, dry north)
************************************************************

  AGRITECH AI — FARMER ADVISORY REPORT
  Powered by Daniel Oyanogbezina's AgriTech AI Platform

  CROP       : Maize
  FARM SIZE  : 2.0 hectares

------------------------------------------------------------
  YIELD PREDICTION
------------------------------------------------------------
  Predicted yield   : 12.1 tons
  Est. income (low) : N   1,815,000
  Est. income (mid) : N   2,420,000
  Est. income (high): N   3,025,000

------------------------------------------------------------
  PERSONALISED FARMING ADVICE
------------------------------------------------------------
  Season    : Plant: March-April (1st) or July-August (2nd)
  Spacing   : Spacing: 75cm x 25cm (2 seeds per hole)
  Harvest   : Harvest: 90-120 days after planting
  Fertilizer: NPK at planting + Urea top-dress at 6 weeks
  Water     : Supplement 

In [6]:
# =============================================================
# CROP CALENDAR
# Tells farmers what to do THIS MONTH for each crop
# Incredibly practical — no other free tool does this
# =============================================================

import datetime

CROP_CALENDAR = {
    'Cassava': {
        1:  'Prepare land, clear and till. Source improved cuttings.',
        2:  'Continue land prep. Apply pre-planting fertilizer.',
        3:  'PLANT. Use 25-30cm cuttings. Space 1m x 1m.',
        4:  'First weeding. Monitor for whitefly (mosaic vector).',
        5:  'Apply NPK fertilizer. Second weeding.',
        6:  'Monitor disease. Remove infected plants immediately.',
        7:  'Weed if needed. Canopy should be closing.',
        8:  'SECOND SEASON PLANTING. Inspect for stem borers.',
        9:  'Apply foliar spray if needed.',
        10: 'Begin harvesting early-maturing varieties.',
        11: 'Peak harvest season. Negotiate with processors.',
        12: 'Harvest and process. Best prices this month.',
    },
    'Maize': {
        1:  'Plan season. Source certified seeds.',
        2:  'Prepare land. Test soil pH.',
        3:  'PLANT first season. Apply basal NPK fertilizer.',
        4:  'First weeding. Watch for Fall Armyworm (inspect nightly).',
        5:  'Top-dress with Urea. Critical tasselling period.',
        6:  'Harvest first season. Dry grain to 13% moisture.',
        7:  'PLANT second season immediately after harvest.',
        8:  'Weed and fertilize second season crop.',
        9:  'Watch for disease. Apply fungicide if needed.',
        10: 'Harvest second season. Store properly.',
        11: 'BEST PRICES this month. Sell stored grain now.',
        12: 'Market and plan next year. Rotate with legume crop.',
    },
    'Yam': {
        1:  'PLANT. Use 200-500g seed yams. Mound 1m apart.',
        2:  'Stake plants as they emerge. First weeding.',
        3:  'Apply NPK. Ensure stakes are firm.',
        4:  'Monitor for mealybugs and beetles.',
        5:  'Top-dress fertilizer. Mulch around mounds.',
        6:  'Weed carefully — roots are near surface.',
        7:  'Monitor tuber development. Do not harvest yet.',
        8:  'Begin harvest of early varieties. NEW YAM FESTIVAL.',
        9:  'PEAK HARVEST. Best prices of the year.',
        10: 'Final harvest. Cure tubers in shade 1-2 weeks.',
        11: 'Store cured yams. Inspect weekly for rot.',
        12: 'Market stored yams. Plan next season seed yams.',
    },
    'Rice': {
        1:  'Best prices for stored rice. Sell now.',
        2:  'Prepare paddy fields. Repair bunds and channels.',
        3:  'Source certified seeds. Prepare nursery beds.',
        4:  'Nursery planting. Soak seeds 24hrs before sowing.',
        5:  'TRANSPLANT to main field. Apply basal fertilizer.',
        6:  'Maintain water level. First Urea application.',
        7:  'Second Urea at tillering. Manage weeds in field.',
        8:  'Panicle initiation — maintain water level.',
        9:  'Monitor for blast disease. Drain field 2 wks pre-harvest.',
        10: 'HARVEST when 80% of grains turn golden.',
        11: 'Dry and mill. Store in sealed bags.',
        12: 'Market. Prices rising — consider holding stock.',
    },
    'Groundnut': {
        1:  'Prepare land. Source certified seeds.',
        2:  'Till to 20cm depth. Apply phosphorus fertilizer.',
        3:  'Wait for rains.',
        4:  'PLANT with first reliable rains. Space 45x15cm.',
        5:  'First weeding. Earth up around plants.',
        6:  'Watch for Rosette virus. Remove infected plants.',
        7:  'Stop weeding — pegs entering soil. Do not disturb.',
        8:  'Monitor maturity. Leaves yellowing = nearly ready.',
        9:  'HARVEST. Dry in windrows for 2 weeks.',
        10: 'Shell and grade. Sell premium grade.',
        11: 'BEST PRICES November-December.',
        12: 'Market remaining stock.',
    },
    'Cocoa': {
        1:  'Prune trees. Remove dead wood and mistletoe.',
        2:  'Apply potassium fertilizer.',
        3:  'Monitor for Black Pod. Apply copper fungicide.',
        4:  'MAIN PLANTING SEASON for new farms.',
        5:  'Fertilize established farms.',
        6:  'Harvest mid-crop pods. Ferment 6 days.',
        7:  'Monitor Black Pod intensively (wet season).',
        8:  'Continue mid-crop harvest. Apply fungicide.',
        9:  'Prepare for main crop. Clear harvesting paths.',
        10: 'MAIN CROP starts. Harvest every 2 weeks.',
        11: 'Peak main crop harvest. Ferment and dry carefully.',
        12: 'Final main crop. Grade and sell to co-ops.',
    },
}

def get_monthly_advice(crop, month=None):
    """Returns what a farmer should do this month"""
    if month is None:
        month = datetime.datetime.now().month

    crop = crop.strip().title()
    if crop not in CROP_CALENDAR:
        return f"Crop '{crop}' not in calendar."

    month_names = [
        '', 'January', 'February', 'March', 'April',
        'May', 'June', 'July', 'August', 'September',
        'October', 'November', 'December'
    ]

    current  = CROP_CALENDAR[crop][month]
    prev_m   = 12 if month == 1 else month - 1
    next_m   = 1  if month == 12 else month + 1
    previous = CROP_CALENDAR[crop][prev_m]
    upcoming = CROP_CALENDAR[crop][next_m]

    print(f"\n{'='*60}")
    print(f"  CROP CALENDAR: {crop.upper()}")
    print(f"{'='*60}")
    print(f"\n  Last month ({month_names[prev_m]}):")
    print(f"  -> {previous}")
    print(f"\n  THIS MONTH ({month_names[month]}) — ACTION REQUIRED:")
    print(f"  -> {current}")
    print(f"\n  Next month ({month_names[next_m]}) — prepare for:")
    print(f"  -> {upcoming}")
    print(f"\n{'='*60}")


# Test all crops
current_month = datetime.datetime.now().month
print(f"Crop calendar for all crops this month:\n")

for crop in CROP_CALENDAR.keys():
    get_monthly_advice(crop, current_month)

Crop calendar for all crops this month:


  CROP CALENDAR: CASSAVA

  Last month (August):
  -> SECOND SEASON PLANTING. Inspect for stem borers.

  THIS MONTH (September) — ACTION REQUIRED:
  -> Apply foliar spray if needed.

  Next month (October) — prepare for:
  -> Begin harvesting early-maturing varieties.


  CROP CALENDAR: MAIZE

  Last month (August):
  -> Weed and fertilize second season crop.

  THIS MONTH (September) — ACTION REQUIRED:
  -> Watch for disease. Apply fungicide if needed.

  Next month (October) — prepare for:
  -> Harvest second season. Store properly.


  CROP CALENDAR: YAM

  Last month (August):
  -> Begin harvest of early varieties. NEW YAM FESTIVAL.

  THIS MONTH (September) — ACTION REQUIRED:
  -> PEAK HARVEST. Best prices of the year.

  Next month (October) — prepare for:
  -> Final harvest. Cure tubers in shade 1-2 weeks.


  CROP CALENDAR: RICE

  Last month (August):
  -> Panicle initiation — maintain water level.

  THIS MONTH (September) — ACTION R

In [7]:
print("""
=============================================================
  DAY 4 REFLECTION
=============================================================

  What I built today:
  [x] Loaded Day 3 saved model without retraining
  [x] Built a complete advisory knowledge base
      encoding real Nigerian agronomy
  [x] Prediction engine with risk flags
  [x] What-if analysis (what if farmer upgrades inputs?)
  [x] Income range estimates (low / mid / high)
  [x] Full crop calendar for 6 crops x 12 months
  [x] Monthly action advice for every crop
  [x] Tested on 4 real Nigerian farmer profiles

  What this means for the product:
  A farmer can now get in 5 seconds what used to
  require an extension officer visit — which could
  take days or weeks in rural Nigeria.

  Key lesson today:
  Domain knowledge + code = real product.
  Your agroforestry MSc is not separate from your AI work.
  It IS your AI work. You encoded years of agricultural
  science into a system anyone can use.

  Tomorrow — Day 5:
  -> Build a simple Flask web API around this tool
  -> Any app (phone, web, WhatsApp) can call it
  -> First step toward farmers using it on their phones

  Day 4 of 365 complete.
=============================================================
""")


  DAY 4 REFLECTION

  What I built today:
  [x] Loaded Day 3 saved model without retraining
  [x] Built a complete advisory knowledge base
      encoding real Nigerian agronomy
  [x] Prediction engine with risk flags
  [x] What-if analysis (what if farmer upgrades inputs?)
  [x] Income range estimates (low / mid / high)
  [x] Full crop calendar for 6 crops x 12 months
  [x] Monthly action advice for every crop
  [x] Tested on 4 real Nigerian farmer profiles

  What this means for the product:
  A farmer can now get in 5 seconds what used to
  require an extension officer visit — which could
  take days or weeks in rural Nigeria.

  Key lesson today:
  Domain knowledge + code = real product.
  Your agroforestry MSc is not separate from your AI work.
  It IS your AI work. You encoded years of agricultural
  science into a system anyone can use.

  Tomorrow — Day 5:
  -> Build a simple Flask web API around this tool
  -> Any app (phone, web, WhatsApp) can call it
  -> First step toward f